In [64]:
from pathlib import Path
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from sklearn.model_selection import train_test_split
from peft import LoraConfig, TaskType
from torch.utils.data import Dataset

random_state = 2026

In [66]:
def generate_dataset(path="./diabetes.csv"):
  df = pd.read_csv(path)
  df = df.drop(columns=["Gender", "HDL", "FFPG", "ALT", "BUN", "CCR"])
  rename_cols = {
      "Age": "age",
      "BMI": "bmi",
      "SBP": "systolic_blood_pressure",
      "DBP": "diastolic_blood_pressure",
      "Chol": "total_cholesterol",
      "FPG": "fasting_plasma_glucose",
      "Tri": "triglycerides",
      "LDL": "low_density_lipoprotein",
      "family_histroy": "family_history",
      "Diabetes": "diabetes"
  }

  df = df.rename(columns=rename_cols)

  df.drop

  df['smoking'] = df['smoking'].astype('string')
  df.loc[df.smoking == "1.0", "smoking"] = "Current Smoker"
  df.loc[df.smoking == "2.0", "smoking"] = "Ever Smoker"
  df.loc[df.smoking == "3.0", "smoking"] = "Never Smoker"

  df['drinking'] = df['drinking'].astype('string')
  df.loc[df.drinking == "1.0", "drinking"] = "Current Drinker"
  df.loc[df.drinking == "2.0", "drinking"] = "Ever Drinker"
  df.loc[df.drinking == "3.0", "drinking"] = "Never Drinker"

  df['family_history'] = df['family_history'].astype('string')
  df.loc[df.family_history == "1", "family_history"] = "Yes"
  df.loc[df.family_history == "0", "family_history"] = "No"

  df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)
  return df


In [67]:
df = generate_dataset()
df.iloc[3]

,3
age,77.000000
bmi,24.300000
systolic_blood_pressure,141.000000
diastolic_blood_pressure,69.000000
fasting_plasma_glucose,6.900000
total_cholesterol,4.760000
triglycerides,0.890000
low_density_lipoprotein,4.860753
smoking,4.860753
drinking,4.860753


In [6]:
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1")
model = AutoModelForSequenceClassification.from_pretrained("dmis-lab/biobert-base-cased-v1.1", num_labels=2, dtype=torch.bfloat16)

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

In [7]:
tokenizer.pad_token = tokenizer.eos_token
model.to("cuda")

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [63]:
temp_df = df.head(50)

y = temp_df["diabetes"]
X = temp_df.drop(columns=["diabetes"])

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=random_state, shuffle=False)

11

In [ ]:
class